# DeBERTa-v3-large Baseline Training

Train baseline model on LIAR dataset (6-class classification).

## Configuration
- **Model**: microsoft/deberta-v3-large
- **Optimizer**: AdamW (lr=8e-6)
- **Scheduler**: Cosine with warmup
- **Loss**: Weighted Cross-Entropy + Label Smoothing

In [ ]:
# Imports
import os, math, random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, set_seed, get_cosine_schedule_with_warmup
)
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 42  #42,123,456
set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load and Prepare Dataset

In [ ]:
# Load dataset
ds = load_dataset('ucsbnlp/liar')
print(ds)

# Label config
LABELS = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

def safe_str(x):
    if x is None: return ''
    if isinstance(x, (list, tuple)):
        return ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject = safe_str(example.get('subjects', example.get('subject', '')))
    context = safe_str(example.get('context', ''))
    parts = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    if isinstance(y, str):
        example['label_id'] = label2id[y.strip().lower()]
    else:
        example['label_id'] = int(y)
    return example

ds = ds.map(build_text)

C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\datasets\load.py:1429: FutureWarning: The repository for ucsbnlp/liar contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ucsbnlp/liar
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 10269
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1283
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1284
    })
})


## 2. Tokenization

In [3]:
MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tokenized = ds.map(tok, batched=True)
tokenized = tokenized.remove_columns([c for c in tokenized['train'].column_names 
                                       if c not in ('input_ids','attention_mask','label_id')])
tokenized = tokenized.rename_column('label_id', 'labels')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized)

C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 10269
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1283
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1284
    })
})


C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\transformers\convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## 3. Class Weights

In [4]:
# Calculate class weights
train_labels = np.array(tokenized['train']['labels'])
counts = np.bincount(train_labels, minlength=len(LABELS)).astype(np.float32)
freq = counts / counts.sum()
inv = 1.0 / (freq + 1e-9)
weights = inv / inv.mean()
class_weights = torch.tensor(weights, dtype=torch.float32)

print('Class weights:')
for i, (label, w) in enumerate(zip(LABELS, weights)):
    print(f'  {label}: {w:.3f}')

Class weights:
  pants-fire: 0.777
  false: 0.731
  barely-true: 0.790
  half-true: 0.922
  mostly-true: 0.937
  true: 1.843


## 4. Model and Custom Trainer

In [5]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Custom trainer with weighted loss
LABEL_SMOOTHING = 0.05

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**{k:v for k,v in inputs.items() if k != 'labels'})
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device), label_smoothing=LABEL_SMOOTHING
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
    }

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Training Configuration

In [ ]:
OUTPUT_DIR = 'baseline_42'

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    num_train_epochs=6,
    learning_rate=8e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=SEED,
)

# Optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)

train_len = len(tokenized['train'])
steps_per_epoch = math.ceil(train_len / (args.per_device_train_batch_size * args.gradient_accumulation_steps))
num_training_steps = steps_per_epoch * int(args.num_train_epochs)
num_warmup_steps = int(args.warmup_ratio * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f'Training steps: {num_training_steps}, Warmup: {num_warmup_steps}')

Training steps: 1926, Warmup: 192


## 6. Train Model

In [7]:
trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    optimizers=(optimizer, scheduler),
)

train_out = trainer.train()
print(f"Training complete! Loss: {train_out.metrics['train_loss']:.4f}")

C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\accelerate\accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.781900,1.751506,0.244548,0.168867,0.168304
2,1.685000,1.671842,0.278037,0.269903,0.252604
3,1.561400,1.661812,0.283489,0.273250,0.249939
4,1.406600,1.754640,0.297508,0.302568,0.288781
5,1.254000,1.856136,0.306075,0.309983,0.300954
6,1.202400,1.872117,0.308411,0.315056,0.304765


C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:

Training complete! Loss: 1.4886


## 7. Evaluate

In [8]:
# Evaluate
val_metrics = trainer.evaluate(tokenized['validation'])
test_metrics = trainer.evaluate(tokenized['test'])

print(f"\nValidation F1: {val_metrics['eval_f1_macro']:.4f}")
print(f"Test F1: {test_metrics['eval_f1_macro']:.4f}")

# Detailed report
pred = trainer.predict(tokenized['test'])
preds = np.argmax(pred.predictions, axis=-1)
print('\n' + classification_report(pred.label_ids, preds, target_names=LABELS, digits=4))


Validation F1: 0.3151
Test F1: 0.3187

              precision    recall  f1-score   support

  pants-fire     0.3520    0.2520    0.2937       250
       false     0.2500    0.2584    0.2541       267
 barely-true     0.3118    0.2129    0.2530       249
   half-true     0.3048    0.5071    0.3808       211
 mostly-true     0.2978    0.3131    0.3052       214
        true     0.4512    0.4022    0.4253        92

    accuracy                         0.3087      1283
   macro avg     0.3279    0.3243    0.3187      1283
weighted avg     0.3133    0.3087    0.3032      1283



## 8. Save Model

In [ ]:
FINAL_DIR = 'baseline_42'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Model saved to {FINAL_DIR}')

Model saved to final_model_baseline_456
